# Demo — end-to-end GNN–BERT context inference

Load one held-out clip, **torch.load a preprocessed .pt graph**, draw it, and run the Task 3 fusion model.

In [ ]:
import json, sys
from pathlib import Path
import numpy as np
import torch
import matplotlib.pyplot as plt
import networkx as nx

ROOT = Path('..').resolve() if Path.cwd().name == 'notebooks' else Path('.').resolve()
sys.path.insert(0, str(ROOT))

from src.bert_encoder import Vocab
from src.config import load_config, resolve_device
from src.evaluate import reconstruct_models
from src.synthetic import load_cached_corpus
from src.taxonomy import GENRES, TAGS

cfg = load_config()
device = resolve_device(cfg)
tracks = {t['id']: t for t in load_cached_corpus()}
splits = json.loads((ROOT / 'data/splits/splits.json').read_text())
tid = splits['test'][0]
t = tracks[tid]
print(tid, t['title'], t['genre'])
print(t['caption'])

In [ ]:
# Preprocessed graph samples are .pt dicts. Load with torch.load:
from src.load_graph import draw_pt_graph, graph_to_networkx, load_pt_graph

pt_path = ROOT / "data/processed/graphs/track_000_chord.pt"
graph = torch.load(pt_path, weights_only=False)
print(graph.keys())
print(graph["node_labels"])

# Same helper the lab UI uses
blob = load_pt_graph(pt_path)
G = graph_to_networkx(blob)
print("nodes", G.number_of_nodes(), "edges", G.number_of_edges(), "kind", blob["kind"])
draw_pt_graph(blob, title="track_000_chord.pt")
plt.show()

seg = load_pt_graph(ROOT / "data/processed/graphs/track_000_segment.pt")
print("segment labels", seg["node_labels"])
draw_pt_graph(seg, title="track_000_segment.pt")
plt.show()

In [ ]:
vocab = Vocab.load(ROOT / 'data/processed/vocab.json')
in_dim = t['seg_x'].shape[1]
models = reconstruct_models(cfg, vocab, in_dim, device)
x = torch.from_numpy(np.asarray(t['seg_x'], np.float32)).unsqueeze(0).to(device)
adj = torch.from_numpy(np.asarray(t['seg_adj'], np.float32)).unsqueeze(0).to(device)
mask = torch.ones(1, x.shape[1], device=device)
ids = torch.from_numpy(vocab.encode(t['caption'], int(cfg['max_text_len']))).unsqueeze(0).to(device)
with torch.no_grad():
    out = models['fusion_cross_attention'](x, adj, mask, ids)
    bert_logits = models['bert'](ids)[0]
    genre_logits = models['gnn_genre'](x, adj, mask)[0]
p = torch.sigmoid(out['tag_logits'][0]).cpu().numpy()
p_b = torch.sigmoid(bert_logits).cpu().numpy()
print('true genre', t['genre'], 'GNN genre', GENRES[int(genre_logits.argmax())])
print('true tags', t['tag_names'])
print('fusion tags', [TAGS[i] for i in np.where(p >= 0.5)[0]])
print('BERT-only  ', [TAGS[i] for i in np.where(p_b >= 0.5)[0]])
print('valence true/pred', round(t['valence'], 3), round(float(out['valence'][0]), 3))
print('arousal true/pred', round(t['arousal'], 3), round(float(out['arousal'][0]), 3))

Attention over caption tokens (Task 3 cross-attention weights).

In [ ]:
from src.bert_encoder import tokenize, CLS, SEP, PAD
attn = out['attn'][0].cpu().numpy() if out.get('attn') is not None else None
tokens = ['[CLS]'] + tokenize(t['caption'])[: int(cfg['max_text_len']) - 2] + ['[SEP]']
if attn is not None:
    w = attn[: len(tokens)]
    plt.figure(figsize=(min(12, 0.45 * len(tokens) + 2), 2.2))
    plt.bar(range(len(tokens)), w, color='#1f8a7a')
    plt.xticks(range(len(tokens)), tokens, rotation=75, ha='right', fontsize=8)
    plt.ylabel('attn')
    plt.tight_layout()
    plt.show()